In [34]:
import jax
import flax.nnx as nnx
from pathlib import Path

from helper import MadLLM, generate_story

In [35]:
model = MadLLM()

In [36]:
model

MadLLM( # Param: 20,212,608 (80.9 MB)
  maxlen=128,
  embedding=TokenAndPositionEmbedding( # Param: 9,673,920 (38.7 MB)
    token_emb=Embed( # Param: 9,649,344 (38.6 MB)
      embedding=Param( # 9,649,344 (38.6 MB)
        value=Array(shape=(50257, 192), dtype=dtype('float32'))
      ),
      num_embeddings=50257,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      promote_dtype=<function promote_dtype at 0x000001ACD1322CA0>
    ),
    pos_emb=Embed( # Param: 24,576 (98.3 KB)
      embedding=Param( # 24,576 (98.3 KB)
        value=Array(shape=(128, 192), dtype=dtype('float32'))
      ),
      num_embeddings=128,
      features=192,
      dtype=dtype('float32'),
      param_dtype=float32,
      promote_dtype=<function promote_dtype at 0x000001ACD1322CA0>
    )
  ),
  transformer_blocks=List([
    TransformerBlock( # Param: 148,224 (592.9 KB)
      attention=MultiHeadAttention( # Param: 148,224 (592.9 KB)
        num_heads=6,
        in_features=192,
      

In [37]:
#load the saved checkpoint
import orbax
from orbax import checkpoint

from jax.sharding import SingleDeviceSharding 

In [38]:
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)

restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)

In [39]:
nnx.state(model)

State({
  'embedding': {
    'pos_emb': {
      'embedding': Param( # 24,576 (98.3 KB)
        value=Array([[ 0.01653565,  0.01149229,  0.0628325 , ..., -0.00773051,
                 0.0018737 , -0.0549496 ],
               [-0.01241376,  0.01329277,  0.08858206, ..., -0.02102038,
                 0.05572481,  0.02097239],
               [ 0.05837216, -0.03376462,  0.03400452, ...,  0.12310911,
                -0.02381788, -0.04554714],
               ...,
               [ 0.10860952,  0.02204411,  0.03419787, ...,  0.01394619,
                -0.01331562,  0.06901263],
               [ 0.14231183, -0.0227183 ,  0.01450266, ..., -0.12847306,
                -0.01344753, -0.01756989],
               [ 0.04195637, -0.08212744, -0.01975309, ...,  0.05484168,
                 0.00886467, -0.03613013]], dtype=float32)
      )
    },
    'token_emb': {
      'embedding': Param( # 9,649,344 (38.6 MB)
        value=Array([[ 0.09022879,  0.05896454,  0.03221416, ...,  0.04800711,
              

In [40]:
checkpoint_path = Path.cwd() / "small_checkpoint.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()

In [41]:
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model,restored_state)

running model

In [42]:
def create_story(story_prompt, temperature, max_new_tokens):
    return generate_story(model, story_prompt, temperature, max_new_tokens)

In [43]:
create_story("Once upon a time a big bear ", 0.2, 30)

'Once upon a time a big bear  best friends, there was a little girl named Lily. She loved to play with her friends. One day, she saw a big tree. The bird'

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=create_story,
    inputs=[
        gr.Textbox(label="Story Prompt"),         
        gr.Slider(
            minimum=0, maximum=1.0, value=0.8, step=0.01, label="Temperature"
        ),
        gr.Slider(minimum=0, maximum=200, value=10, step=1, label="Max Tokens"
        )
    ],
    outputs=["text"]
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://aac9f13a1257d419f8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Created dataset file at: .gradio\flagged\dataset1.csv


d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\VS code projects\AI\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
